In [ ]:
import torch, torch_geometric
import pandas as pd
import numpy as np

In [ ]:
print(torch.__version__)
print(torch.backends.mps.is_available())
print(torch_geometric.__version__)

In [ ]:
from lib.data_management import load_match_tables_jsonl

In [ ]:
data_path = "../data/pro_match_tables.jsonl"

matches = load_match_tables_jsonl(data_path)

matches[:10]

In [ ]:
match = matches[0]
match.outcome

In [ ]:
outcome_df = pd.DataFrame([m.outcome.model_dump() for m in matches if m.outcome is not None])
outcome_df

In [ ]:
df = pd.DataFrame([match.model_dump() for match in matches])
df

In [ ]:
combined_df = df.merge(outcome_df, on="match_id", how='inner')

In [ ]:
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from torch import nn
from torch_geometric.nn.aggr import AttentionalAggregation 
from torch.utils.data import random_split

from torch_geometric.nn import RGCNConv

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import random_split
from sklearn.model_selection import train_test_split # Import for splitting

# PyTorch Geometric imports
from torch_geometric.nn import GCNConv, global_mean_pool, global_add_pool, global_max_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

def get_default_device():
    """Pick GPU if available, else CPU."""
    if torch.cuda.is_available():
        return torch.device('cuda')
    elif torch.backends.mps.is_available():
        return torch.device('mps')
    else:
        return torch.device('cpu')


# --- Step 1: Data Preparation Function (UPDATED) ---
def prepare_draft_arrays(df: pd.DataFrame, id_to_idx_map: dict = None):
    """
    Processes a raw DataFrame into numpy arrays for graph creation.
    If id_to_idx_map is provided, it uses it (for test/validation data).
    If not, it creates a new one (for training data).
    """
    rad_cols = [f"slot_{i}_hero_id" for i in range(5)]
    dir_cols = [f"slot_{i}_hero_id" for i in range(128, 133)]
    
    sub = df.dropna(subset=rad_cols + dir_cols).copy()
    sub.reset_index(drop=True, inplace=True)
    
    radiant_ids_raw = sub[rad_cols].astype(np.int64).to_numpy()
    dire_ids_raw = sub[dir_cols].astype(np.int64).to_numpy()
    
    # === NEW LOGIC: Handle training vs. testing mapping ===
    create_map = id_to_idx_map is None
    if create_map:
        print("Preparing TRAIN data: Creating new hero ID to index map.")
        all_ids = np.concatenate([radiant_ids_raw.reshape(-1), dire_ids_raw.reshape(-1)])
        uniq = np.unique(all_ids)
        id_to_idx_map = {int(h): i for i, h in enumerate(uniq)}
    else:
        print("Preparing TEST data: Using existing hero ID to index map.")

    num_heroes = len(id_to_idx_map)
    
    # When testing, we need to handle heroes that were not in the training set.
    # We will filter out matches containing unknown heroes.
    all_known_heroes = set(id_to_idx_map.keys())
    valid_rows_mask = np.array([
        all(hero in all_known_heroes for hero in row) 
        for row in np.concatenate([radiant_ids_raw, dire_ids_raw], axis=1)
    ])
    
    if np.sum(~valid_rows_mask) > 0:
        print(f"WARNING: Dropping {np.sum(~valid_rows_mask)} matches from the dataset "
              f"because they contain heroes not seen during training.")
        sub = sub[valid_rows_mask].reset_index(drop=True)
        radiant_ids_raw = radiant_ids_raw[valid_rows_mask]
        dire_ids_raw = dire_ids_raw[valid_rows_mask]
    
    print(f"Processed {len(sub)} valid matches.")
    
    mapper = np.vectorize(id_to_idx_map.get, otypes=[np.int64])
    radiant_ids = mapper(radiant_ids_raw)
    dire_ids = mapper(dire_ids_raw)
    
    if "radiant_win" in sub.columns:
        y = sub["radiant_win"].astype(int).to_numpy()
    else:
        raise ValueError("No 'radiant_win' label column found.")
        
    if create_map:
        return radiant_ids, dire_ids, y, num_heroes, id_to_idx_map
    else:
        # Don't return the map when we're just using it
        return radiant_ids, dire_ids, y, num_heroes


def build_relational_edges_for_10_nodes():
    """
    Builds a static edge structure with three relation types:
    0: Synergy (same team)
    1: Counter (opposite team)
    2: Self-loop (identity)
    """
    radiant_nodes = list(range(5))
    dire_nodes = list(range(5, 10))
    
    edges = []
    edge_types = []
    
    # Relation 0: Synergy Edges
    for team in (radiant_nodes, dire_nodes):
        for i in team:
            for j in team:
                if i != j:
                    edges.append((i, j))
                    edge_types.append(0) # Synergy type
                    
    # Relation 1: Counter Edges
    for i in radiant_nodes:
        for j in dire_nodes:
            # Add edges in both directions
            edges.append((i, j))
            edge_types.append(1) # Counter type
            
            edges.append((j, i))
            edge_types.append(1) # Counter type
            
    # Relation 2: Self-Loop Edges
    for i in range(10):
        edges.append((i, i))
        edge_types.append(2) # Self-loop type
        
    edge_index = torch.tensor(edges, dtype=torch.long).t().contiguous()
    edge_type = torch.tensor(edge_types, dtype=torch.long)
    
    return edge_index, edge_type

# Create these as global constants
RELATIONAL_EDGE_INDEX, RELATIONAL_EDGE_TYPE = build_relational_edges_for_10_nodes()


def make_relational_graphs_from_arrays(r_ids_np, d_ids_np, y_np):
    data_list = []
    for r5, d5, y in zip(r_ids_np, d_ids_np, y_np):
        hero_indices = torch.tensor(np.concatenate([r5, d5]), dtype=torch.long)
        y_tensor = torch.tensor([int(y)], dtype=torch.long)
        
        # Use the new relational edge attributes
        g = Data(
            x=hero_indices,
            edge_index=RELATIONAL_EDGE_INDEX,
            edge_type=RELATIONAL_EDGE_TYPE, # Add the edge types
            y=y_tensor
        )
        g.num_nodes = 10
        data_list.append(g)
    return data_list


class RelationalGCN(nn.Module):
    def __init__(self, num_heroes, num_relations, emb_dim, hidden_dim, dropout_rate=0.5):
        super().__init__()
        
        self.num_relations = num_relations
        
        self.embedding = nn.Embedding(num_heroes, emb_dim)
        
        self.conv1 = RGCNConv(emb_dim, hidden_dim, num_relations)
        self.conv2 = RGCNConv(hidden_dim, hidden_dim, num_relations)

        # Team-aware readout: pool Radiant and Dire separately, then combine
        # Input is 4 * hidden_dim: [radiant_pool, dire_pool, diff, interaction]
        self.output_layer = nn.Sequential(
            nn.Linear(4 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_dim, 1),
        )
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, data: Data) -> torch.Tensor:
        x, edge_index, edge_type, batch = data.x, data.edge_index, data.edge_type, data.batch
        
        # 1. Get hero embeddings
        x = self.embedding(x)
        
        # 2. Message passing
        x = self.conv1(x, edge_index, edge_type)
        x = F.relu(x)
        x = self.dropout(x)
        
        x = self.conv2(x, edge_index, edge_type)
        x = F.relu(x)
        x = self.dropout(x)
        
        # 3. Team-aware readout (FIX: was global_add_pool over all 10 nodes)
        # Each graph has exactly 10 nodes: [R0,R1,R2,R3,R4, D0,D1,D2,D3,D4]
        num_graphs = batch.max().item() + 1
        x_per_graph = x.view(num_graphs, 10, -1)
        
        radiant_pool = x_per_graph[:, :5, :].mean(dim=1)
        dire_pool = x_per_graph[:, 5:, :].mean(dim=1)
        
        diff = radiant_pool - dire_pool
        interaction = radiant_pool * dire_pool
        
        combined = torch.cat([radiant_pool, dire_pool, diff, interaction], dim=1)
        
        # 4. Final prediction
        logits = self.output_layer(combined)
        return logits.squeeze(1)

In [ ]:
from torch.utils.data import Dataset

class DotaDraftDataset(Dataset):
    def __init__(self, radiant_ids, dire_ids, y):
        self.radiant_ids = radiant_ids
        self.dire_ids = dire_ids
        self.y = y

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        # Fetch data for a single match
        r5_id = self.radiant_ids[idx]
        d5_id = self.dire_ids[idx]
        y_val = self.y[idx]

        # Create the graph object just-in-time
        hero_indices = torch.tensor(np.concatenate([r5_id, d5_id]), dtype=torch.long)
        y_tensor = torch.tensor([int(y_val)], dtype=torch.long)
        
        g = Data(
            x=hero_indices,
            edge_index=RELATIONAL_EDGE_INDEX, # Assumes this is a global constant
            edge_type=RELATIONAL_EDGE_TYPE,   # Assumes this is a global constant
            y=y_tensor
        )
        g.num_nodes = 10
        return g

In [ ]:
class GNNTrainer:
    def __init__(self, num_heroes: int, emb_dim: int, hidden_dim: int, lr: float, dropout_rate: float, weight_decay: float = 0.0):
        self.device = get_default_device() # Assuming get_default_device is defined
        print(f"GNNTrainer using device: {self.device}")

        self.model = RelationalGCN(
            num_heroes=num_heroes, num_relations=3, emb_dim=emb_dim, hidden_dim=hidden_dim, dropout_rate=dropout_rate
        ).to(self.device)
        
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr)
        self.criterion = nn.BCEWithLogitsLoss()
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)

    # _train_one_epoch and _evaluate methods are correct and do not need changes.
    def _train_one_epoch(self, loader: DataLoader) -> float:
        self.model.train()
        total_loss = 0
        for batch in loader:
            batch = batch.to(self.device)
            self.optimizer.zero_grad()
            logits = self.model(batch)
            loss = self.criterion(logits, batch.y.float())
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item() * batch.num_graphs
        return total_loss / len(loader.dataset)

    @torch.no_grad()
    def _evaluate(self, loader: DataLoader) -> float:
        self.model.eval()
        correct = 0
        for batch in loader:
            batch = batch.to(self.device)
            logits = self.model(batch)
            preds = (logits > 0).long()
            correct += (preds.cpu() == batch.y.cpu()).sum().item()
        return correct / len(loader.dataset)

    def fit(self, radiant_ids, dire_ids, y, epochs=20, batch_size=2048, val_ratio=0.2):
        print("\n--- Starting Training with RelationalGCN ---")
        full_dataset = DotaDraftDataset(radiant_ids, dire_ids, y)
        
        n_val = int(len(full_dataset) * val_ratio)
        n_train = len(full_dataset) - n_val
        train_ds, val_ds = random_split(full_dataset, [n_train, n_val])
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_ds, batch_size=batch_size, num_workers=0)
        
        print(f"Training on {len(train_ds)} samples, validating on {len(val_ds)} samples.")

        # --- ADDED FOR CHECKPOINTING ---
        best_val_acc = 0.0
        model_save_path = "best_rgcn_model.pth"
        # -----------------------------

        for epoch in range(epochs):
            train_loss = self._train_one_epoch(train_loader)
            val_acc = self._evaluate(val_loader)
            print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

            # --- ADDED FOR CHECKPOINTING ---
            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(self.model.state_dict(), model_save_path)
                print(f"  -> New best model saved to {model_save_path} with accuracy: {best_val_acc:.4f}")

    def evaluate_on_test_set(self, test_df: pd.DataFrame, id_to_idx: dict, batch_size: int):
        print("\n--- Evaluating on Holdout Test Set ---")
        r_ids, d_ids, y_test, _ = prepare_draft_arrays(test_df, id_to_idx_map=id_to_idx)
        
        # <<< CORRECTED LINE >>>
        # Also call the new function here for the test set
        test_dataset = make_relational_graphs_from_arrays(r_ids, d_ids, y_test)
        
        test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=0)
        test_acc = self._evaluate(test_loader)
        print(f"Final Test Accuracy: {test_acc:.4f}")
        return test_acc

In [ ]:
train_df, test_df = train_test_split(combined_df, test_size=0.2, shuffle=False, random_state=42)
    
# 2. Process the TRAINING data. This creates the essential `id_to_idx` map.
radiant_ids, dire_ids, y, num_heroes, id_to_idx = prepare_draft_arrays(train_df)

# 3. Define hyperparameters
HP = {
    "emb_dim": 128, "hidden_dim": 256, "lr": 1e-4, "dropout_rate": 0.5,
    "epochs": 30, "batch_size": 2048, "weight_decay": 1e-5
}

# 4. Initialize and train the model ONLY on the training data
simple_trainer = GNNTrainer(
    num_heroes=num_heroes, emb_dim=HP["emb_dim"], hidden_dim=HP["hidden_dim"],
    lr=HP["lr"], dropout_rate=HP["dropout_rate"], weight_decay=HP["weight_decay"]
)
simple_trainer.fit(
    radiant_ids, dire_ids, y, epochs=HP["epochs"], batch_size=HP["batch_size"]
)

# 5. Perform final evaluation on the unseen test_df
#    Pass the test data and the crucial mapping from the training step.
simple_trainer.evaluate_on_test_set(test_df, id_to_idx, batch_size=HP["batch_size"])

In [ ]:
from lib.data_management import load_pro_matches_parquet

In [ ]:
pub_patch_fp = "../data/public_matches_dataset.parquet"

pub_matches_df = load_pro_matches_parquet(pub_patch_fp)

In [ ]:
pub_matches_df

In [ ]:
train_df, test_df = train_test_split(pub_matches_df, test_size=0.2, shuffle=False, random_state=42)
    
# 2. Process the TRAINING data. This creates the essential `id_to_idx` map.
radiant_ids, dire_ids, y, num_heroes, id_to_idx = prepare_draft_arrays(train_df)

# 3. Define hyperparameters
HP = {
    "emb_dim": 128, "hidden_dim": 256, "lr": 3e-4, "dropout_rate": 0.3,
    "epochs": 30, "batch_size": 2048, "weight_decay": 1e-5
}

# 4. Initialize and train the model ONLY on the training data
simple_trainer = GNNTrainer(
    num_heroes=num_heroes, emb_dim=HP["emb_dim"], hidden_dim=HP["hidden_dim"],
    lr=HP["lr"], dropout_rate=HP["dropout_rate"], weight_decay=HP["weight_decay"]
)
simple_trainer.fit(
    radiant_ids, dire_ids, y, epochs=HP["epochs"], batch_size=HP["batch_size"]
)

# 5. Perform final evaluation on the unseen test_df
#    Pass the test data and the crucial mapping from the training step.
simple_trainer.evaluate_on_test_set(test_df, id_to_idx, batch_size=HP["batch_size"])

In [25]:
from pathlib import Path
import json
from dota_oracle_common.models.heroes import HeroData

heros_path = Path("../data/heroes_data.jsonl")
with heros_path.open("r", encoding="utf-8") as f:
    heroes = json.load(f)

hero_objects = [HeroData.model_validate(hero) for hero in heroes]
hero_objects[:5]

[HeroData(primary_attr='str', agi_gain=1.4, base_agi=11.0, int_gain=1.8, base_int=16.0, str_gain=3.0, base_str=25.0, attack_type='Melee', attack_rate=1.7, attack_range=175.0, attack_point=0.5, projectile_speed=0.0, base_armor=0.0, base_mana=75, base_mana_regen=0.0, base_health=120, base_health_regen=2.5, id=14, localized_name='Pudge', roles=['Disabler', 'Initiator', 'Durable', 'Nuker'], img='/apps/dota2/images/dota_react/heroes/pudge.png?', icon='/apps/dota2/images/dota_react/heroes/icons/pudge.png?'),
 HeroData(primary_attr='agi', agi_gain=2.8, base_agi=24.0, int_gain=2.2, base_int=21.0, str_gain=2.8, base_str=22.0, attack_type='Ranged', attack_rate=1.7, attack_range=475.0, attack_point=0.3, projectile_speed=2000.0, base_armor=0.0, base_mana=75, base_mana_regen=0.0, base_health=120, base_health_regen=1.0, id=15, localized_name='Razor', roles=['Carry', 'Durable', 'Nuker', 'Pusher'], img='/apps/dota2/images/dota_react/heroes/razor.png?', icon='/apps/dota2/images/dota_react/heroes/icons/

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import gc
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MultiLabelBinarizer
from torch.utils.data import Dataset, random_split

# PyTorch Geometric imports
from torch_geometric.nn import RGCNConv, global_add_pool
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader

# ==============================================================================
# PART 1: FEATURE ENGINEERING (NEW)
# ==============================================================================

# This is the updated, correct version
def create_hero_feature_matrix(hero_objects, id_to_idx_map):
    """
    Transforms a list of Pydantic hero objects into a feature matrix.
    """
    print("Creating rich hero feature matrix from Pydantic objects...")
    
    # Convert the list of objects to a dictionary keyed by hero ID for fast lookup
    hero_data_dict = {h.id: h for h in hero_objects}
    
    num_heroes = len(id_to_idx_map)
    
    # --- 1. Process Numerical Features (using attribute access) ---
    numerical_cols = [
        'agi_gain', 'base_agi', 'int_gain', 'base_int', 'str_gain', 'base_str',
        'attack_rate', 'attack_range', 'attack_point', 'projectile_speed',
        'base_armor', 'base_mana', 'base_mana_regen', 'base_health', 'base_health_regen'
    ]
    numerical_features_list = [[getattr(hero_data_dict[h_id], col) for col in numerical_cols] for h_id in id_to_idx_map.keys()]
    scaler = StandardScaler()
    numerical_features_scaled = scaler.fit_transform(numerical_features_list)
    
    # --- 2. Process Categorical Features (using attribute access) ---
    attr_one_hot = pd.get_dummies([hero_data_dict[h_id].primary_attr for h_id in id_to_idx_map.keys()], prefix='attr').values
    attack_type_one_hot = pd.get_dummies([hero_data_dict[h_id].attack_type for h_id in id_to_idx_map.keys()], prefix='atk').values
    
    # --- 3. Process Roles (using attribute access) ---
    all_roles = [hero_data_dict[h_id].roles for h_id in id_to_idx_map.keys()]
    mlb = MultiLabelBinarizer()
    roles_multi_hot = mlb.fit_transform(all_roles)
    
    combined_features_temp = np.concatenate([
        numerical_features_scaled, attr_one_hot, attack_type_one_hot, roles_multi_hot
    ], axis=1)
    
    feature_dim = combined_features_temp.shape[1]
    print(f"Created hero feature matrix with shape: [{num_heroes}, {feature_dim}]")
    
    final_feature_matrix = np.zeros((num_heroes, feature_dim))
    for hero_id, hero_idx in id_to_idx_map.items():
        original_pos = list(id_to_idx_map.keys()).index(hero_id)
        final_feature_matrix[hero_idx] = combined_features_temp[original_pos]
        
    return torch.tensor(final_feature_matrix, dtype=torch.float)


# ==============================================================================
# PART 2: DATA PIPELINE (UPDATED)
# ==============================================================================

def prepare_draft_arrays(df: pd.DataFrame):
    rad_cols = [f"slot_{i}_hero_id" for i in range(5)]
    dir_cols = [f"slot_{i}_hero_id" for i in range(128, 133)]
    sub = df.dropna(subset=rad_cols + dir_cols).copy().reset_index(drop=True)
    radiant_ids_raw = sub[rad_cols].astype(np.int64).to_numpy()
    dire_ids_raw = sub[dir_cols].astype(np.int64).to_numpy()
    all_ids = np.concatenate([radiant_ids_raw.ravel(), dire_ids_raw.ravel()])
    uniq_ids = np.unique(all_ids)
    id_to_idx = {int(h): i for i, h in enumerate(uniq_ids)}
    num_heroes = len(uniq_ids)
    mapper = np.vectorize(id_to_idx.get, otypes=[np.int64])
    radiant_ids = mapper(radiant_ids_raw)
    dire_ids = mapper(dire_ids_raw)
    y = sub["radiant_win"].astype(int).to_numpy()
    return radiant_ids, dire_ids, y, num_heroes, id_to_idx

def build_relational_edges():
    r, d = list(range(5)), list(range(5, 10))
    edges, types = [], []
    for team in (r, d): # Synergy
        for i in team:
            for j in team:
                if i != j: edges.append((i, j)); types.append(0)
    for i in r: # Counter
        for j in d:
            edges.extend([(i, j), (j, i)]); types.extend([1, 1])
    for i in range(10): # Self-loop
        edges.append((i, i)); types.append(2)
    return torch.tensor(edges, dtype=torch.long).t().contiguous(), torch.tensor(types, dtype=torch.long)

RELATIONAL_EDGE_INDEX, RELATIONAL_EDGE_TYPE = build_relational_edges()

# --- Custom Dataset Class ---
class DotaDraftDataset(Dataset):
    def __init__(self, radiant_ids, dire_ids, y, hero_feature_matrix):
        self.radiant_ids = radiant_ids
        self.dire_ids = dire_ids
        self.y = y
        self.hero_features = hero_feature_matrix

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        hero_indices_np = np.concatenate([self.radiant_ids[idx], self.dire_ids[idx]])
        draft_features = self.hero_features[hero_indices_np]
        y_tensor = torch.tensor([int(self.y[idx])], dtype=torch.long)
        
        g = Data(
            x=draft_features,
            hero_indices=torch.from_numpy(hero_indices_np).long(),
            edge_index=RELATIONAL_EDGE_INDEX,
            edge_type=RELATIONAL_EDGE_TYPE,
            y=y_tensor
        )
        g.num_nodes = 10
        return g

# ==============================================================================
# PART 3: MODEL ARCHITECTURE (FIXED — team-aware pooling)
# ==============================================================================

class HybridGCN(nn.Module):
    def __init__(self, feature_dim, num_heroes, embedding_dim, num_relations, hidden_dim, dropout_rate=0.5):
        super().__init__()
        
        self.hero_embedding = nn.Embedding(num_heroes, embedding_dim)
        
        combined_input_dim = feature_dim + embedding_dim
        
        self.conv1 = RGCNConv(combined_input_dim, hidden_dim, num_relations)
        self.conv2 = RGCNConv(hidden_dim, hidden_dim, num_relations)
        
        # FIX: Team-aware readout head instead of single linear layer
        # Input: [radiant_pool, dire_pool, diff, interaction] = 4 * hidden_dim
        self.output_layer = nn.Sequential(
            nn.Linear(4 * hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate),
            nn.Linear(hidden_dim, 1),
        )
        self.dropout = nn.Dropout(p=dropout_rate)

    def forward(self, data: Data) -> torch.Tensor:
        precomputed_features, hero_indices, edge_index, edge_type, batch = \
            data.x, data.hero_indices, data.edge_index, data.edge_type, data.batch
        
        # 1. Combine static features with learnable embeddings
        learnable_embeddings = self.hero_embedding(hero_indices)
        x = torch.cat([precomputed_features, learnable_embeddings], dim=-1)
        
        # 2. Message passing
        x = self.conv1(x, edge_index, edge_type).relu()
        x = self.dropout(x)
        x = self.conv2(x, edge_index, edge_type).relu()
        x = self.dropout(x)
        
        # 3. FIX: Team-aware readout (was global_add_pool over all 10 nodes)
        # Each graph has exactly 10 nodes: [R0,R1,R2,R3,R4, D0,D1,D2,D3,D4]
        num_graphs = batch.max().item() + 1
        x_per_graph = x.view(num_graphs, 10, -1)
        
        radiant_pool = x_per_graph[:, :5, :].mean(dim=1)
        dire_pool = x_per_graph[:, 5:, :].mean(dim=1)
        
        diff = radiant_pool - dire_pool
        interaction = radiant_pool * dire_pool
        
        combined = torch.cat([radiant_pool, dire_pool, diff, interaction], dim=1)
        
        # 4. Final prediction
        logits = self.output_layer(combined)
        return logits.squeeze(1)

# ==============================================================================
# PART 4: TRAINER CLASS (FIXED — device detection)
# ==============================================================================

class GNNTrainer:
    def __init__(self, feature_dim: int, num_heroes: int, embedding_dim: int, hidden_dim: int, lr: float, dropout_rate: float, weight_decay: float):
        self.device = get_default_device()
        print(f"GNNTrainer using device: {self.device}")

        self.model = HybridGCN(
            feature_dim=feature_dim,
            num_heroes=num_heroes,
            embedding_dim=embedding_dim,
            num_relations=3,
            hidden_dim=hidden_dim,
            dropout_rate=dropout_rate
        ).to(self.device)
        
        self.optimizer = torch.optim.Adam(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        self.criterion = nn.BCEWithLogitsLoss()

    def _train_one_epoch(self, loader: DataLoader) -> float:
        self.model.train()
        total_loss = 0
        for batch in loader:
            batch = batch.to(self.device)
            self.optimizer.zero_grad()
            logits = self.model(batch)
            loss = self.criterion(logits, batch.y.float())
            loss.backward()
            self.optimizer.step()
            total_loss += loss.item() * batch.num_graphs
        return total_loss / len(loader.dataset)

    @torch.no_grad()
    def _evaluate(self, loader: DataLoader) -> float:
        self.model.eval()
        correct = 0
        for batch in loader:
            batch = batch.to(self.device)
            logits = self.model(batch)
            preds = (logits > 0).long()
            correct += (preds.cpu() == batch.y.cpu()).sum().item()
        return correct / len(loader.dataset)

    def fit(self, radiant_ids, dire_ids, y, hero_features, epochs=20, batch_size=2048, val_ratio=0.2):
        print("\n--- Starting Training with Rich Features ---")
        
        full_dataset = DotaDraftDataset(radiant_ids, dire_ids, y, hero_features)
        
        n_val = int(len(full_dataset) * val_ratio)
        n_train = len(full_dataset) - n_val
        train_ds, val_ds = random_split(full_dataset, [n_train, n_val])
        
        train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True, num_workers=0)
        val_loader = DataLoader(val_ds, batch_size=batch_size, num_workers=0)
        
        print(f"Training on {len(train_ds)} samples, validating on {len(val_ds)} samples.")

        best_val_acc = 0.0
        for epoch in range(epochs):
            train_loss = self._train_one_epoch(train_loader)
            val_acc = self._evaluate(val_loader)
            print(f"Epoch {epoch+1:02d}/{epochs} | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.4f}")

            if val_acc > best_val_acc:
                best_val_acc = val_acc
                torch.save(self.model.state_dict(), "best_rich_feature_model.pth")
                print(f"  -> New best model saved with accuracy: {best_val_acc:.4f}")

    def evaluate_on_test_set(self, test_df: pd.DataFrame, id_to_idx: dict, hero_features: torch.Tensor, batch_size: int):
        print("\n--- Evaluating on Holdout Test Set ---")
        r_ids, d_ids, y_test, _, _ = prepare_draft_arrays(test_df)
        
        all_known_heroes = set(id_to_idx.keys())
        valid_rows_mask = np.array([
            all(hero in all_known_heroes for hero in row) 
            for row in np.concatenate([
                test_df[[f"slot_{i}_hero_id" for i in range(5)]].values,
                test_df[[f"slot_{i}_hero_id" for i in range(128, 133)]].values
            ], axis=1)
        ])
        
        test_df_filtered = test_df[valid_rows_mask]
        r_ids, d_ids, y_test, _, _ = prepare_draft_arrays(test_df_filtered)

        test_dataset = DotaDraftDataset(r_ids, d_ids, y_test, hero_features)
        test_loader = DataLoader(test_dataset, batch_size=batch_size, num_workers=0)
        
        print(f"Loading best model from 'best_rich_feature_model.pth'")
        self.model.load_state_dict(torch.load("best_rich_feature_model.pth", weights_only=True))

        test_acc = self._evaluate(test_loader)
        print(f"Final Test Accuracy: {test_acc:.4f}")
        return test_acc

In [35]:
   # --- 2. Split data ---
train_df, test_df = train_test_split(pub_matches_df, test_size=0.2, shuffle=True, random_state=42)

# --- 3. Process matches and create feature matrix ---
radiant_ids, dire_ids, y, num_heroes, id_to_idx = prepare_draft_arrays(train_df)
hero_features = create_hero_feature_matrix(hero_objects, id_to_idx)
feature_dim = hero_features.shape[1]

del train_df
gc.collect()

# --- 4. Define hyperparameters ---
HP = {
        "hidden_dim": 256,
        "embedding_dim": 64, # The size of the learnable part. 64 is a good start.
        "lr": 3e-4, 
        "dropout_rate": 0.4,
        "weight_decay": 1e-5,
        "epochs": 30,
        "batch_size": 2048,
    }

# --- 5. Initialize and train ---
trainer = GNNTrainer(
        feature_dim=feature_dim,
        num_heroes=num_heroes, # Pass this in again
        embedding_dim=HP["embedding_dim"], # Pass in the new hyperparameter
        hidden_dim=HP["hidden_dim"],
        lr=HP["lr"],
        dropout_rate=HP["dropout_rate"],
        weight_decay=HP["weight_decay"]
    )
    
trainer.fit(
        radiant_ids, dire_ids, y,
        hero_features=hero_features,
        epochs=HP["epochs"],
        batch_size=HP["batch_size"]
    )

# --- 6. Final evaluation ---
# The trainer.evaluate_on_test_set method needs a small fix in your original code
# to load the best model before testing. I've added that fix.
trainer.evaluate_on_test_set(
    test_df,
    id_to_idx,
    hero_features,
    batch_size=HP["batch_size"]
)

Creating rich hero feature matrix from Pydantic objects...
Created hero feature matrix with shape: [126, 29]
GNNTrainer using device: mps

--- Starting Training with Rich Features ---
Training on 444474 samples, validating on 111118 samples.
Epoch 01/30 | Train Loss: 0.7129 | Val Acc: 0.5297
  -> New best model saved with accuracy: 0.5297
Epoch 02/30 | Train Loss: 0.6912 | Val Acc: 0.5297
Epoch 03/30 | Train Loss: 0.6913 | Val Acc: 0.5297
Epoch 04/30 | Train Loss: 0.6913 | Val Acc: 0.5297
Epoch 05/30 | Train Loss: 0.6913 | Val Acc: 0.5297
Epoch 06/30 | Train Loss: 0.6913 | Val Acc: 0.5297
Epoch 07/30 | Train Loss: 0.6913 | Val Acc: 0.5297
Epoch 08/30 | Train Loss: 0.6913 | Val Acc: 0.5297
Epoch 09/30 | Train Loss: 0.6913 | Val Acc: 0.5297
Epoch 10/30 | Train Loss: 0.6913 | Val Acc: 0.5297
Epoch 11/30 | Train Loss: 0.6913 | Val Acc: 0.5297
Epoch 12/30 | Train Loss: 0.6913 | Val Acc: 0.5297
Epoch 13/30 | Train Loss: 0.6913 | Val Acc: 0.5297
Epoch 14/30 | Train Loss: 0.6913 | Val Acc: 0.5

KeyboardInterrupt: 